# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rufatj/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (Lane 2 in `docs/ml-intern-dataset-and-lane-guide.md`).

I'm picking this lane because it maps onto a concrete, recurring decision an account team already
faces ("which pages do we open first this week?"), and because the starter pipeline that ships in
this repo (`scripts/01`-`05`) already builds exactly this: a rule baseline, three trained models,
and a ranked review queue with reason codes, all on the small anonymized slice. That gives me a
working, evaluated baseline and a first model from day one (see `outputs/model_report.md` -- random
forest reaches Precision@50 = 0.740 vs. 0.240 for the hand-written rule), so the next seven weeks
are about strengthening the label and the validation on the full warehouse (moving from a
same-window proxy to a real prior-90-days -> next-30-days outcome), not about inventing the
pipeline from nothing.


In [1]:
import os

# Make this notebook runnable from Colab or locally, regardless of cwd.
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "run this from inside the repo"

# Check: pull the baseline-vs-model comparison straight from the already
# committed report, so the claim above is never hand-typed.
report = open("outputs/model_report.md").read()
for line in report.splitlines():
    if line.startswith("| random_forest") or line.startswith("| baseline_rules"):
        print(line)


| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** out of all of a client's existing pages, which ones should a content
editor open and review first this week for a refresh (rewrite, expand, protect, or prune)?

**Who acts, and how:** the FlyRank account strategist (or the client's own editor), who realistically
has time to review a few dozen pages a week per client -- not the full catalog, which runs into the
tens of thousands of pages once you look at the warehouse's 519,606 content items across 104 clients.

**Cost of a wrong call:**
- Ranked too high, not actually worth it (false positive): an editor's scarce review hour is spent
  on a page that was fine, at the cost of a page that genuinely needed attention.
- Ranked too low or missed (false negative): a real, high-demand declining page keeps silently
  losing search visibility and clicks until someone happens to notice it by hand.

Because review time is the scarce resource, what matters is the quality of the *top* of the list,
not overall accuracy -- which is why Precision@K is the metric that matches the decision (see
section 3 of `w02_ml_task_framing.ipynb`), not plain accuracy.

**Why data/ML, and not just a hand-written rule:** a rule baseline already exists in this repo, and
it only reaches ROC AUC 0.627 / Precision@50 0.240 (`outputs/model_report.md`). The signals that
actually separate "worth reviewing" from "fine as is" -- freshness, impression volume, position,
CTR, engagement -- interact non-linearly, and a learned model recovers that pattern well enough to
triple Precision@50 to 0.740 on the same rows. That gap is the evidence this is a real pattern,
just too tangled to hand-code as a simple if-statement.


In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Check: how big is the review-capacity mismatch, even on this small slice?
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(
    f"Pages in this 30,000-row slice alone that are 'stale but still visible' "
    f"(>=180 days since update AND >=500 impressions/90d): {len(stale_visible):,} "
    f"({len(stale_visible) / len(df):.1%} of all pages)."
)
print(
    "That is already more than a handful of editors can review in a week for one slice -- at "
    "full-warehouse scale (519,606 content items) the review-capacity gap is far wider, which is "
    "exactly why the pages need to be ranked, not just listed."
)


Pages in this 30,000-row slice alone that are 'stale but still visible' (>=180 days since update AND >=500 impressions/90d): 17 (0.1% of all pages).
That is already more than a handful of editors can review in a week for one slice -- at full-warehouse scale (519,606 content items) the review-capacity gap is far wider, which is exactly why the pages need to be ranked, not just listed.


## 3. Quick look at the data (2-3 real numbers)

Loaded straight from `data/raw/content_refresh_anonymized.csv` (30,000 pages, 32 clients, trailing
90-day window) -- the numbers below are computed live, not copied from the guide.


In [3]:
declining = df["trend_direction"].eq("down")

print(
    f"1) Declining rate: {declining.sum():,} of {len(df):,} pages have trend_direction == 'down' "
    f"({declining.mean():.1%})."
)

med_declining = df.loc[declining, "impressions_90d"].median()
med_rest = df.loc[~declining, "impressions_90d"].median()
print(
    f"2) Median impressions/90d: {med_declining:,.0f} for declining pages vs. {med_rest:,.0f} for "
    f"the rest -- decline is not confined to low-traffic pages, so there is real demand at stake."
)

declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print(
    f"3) {len(declining_with_demand):,} pages ({len(declining_with_demand) / len(df):.1%}) are "
    f"both declining AND still pulling >=100 impressions/90d -- real, visible demand at risk, not "
    f"noise on pages nobody sees."
)


1) Declining rate: 16,262 of 30,000 pages have trend_direction == 'down' (54.2%).
2) Median impressions/90d: 961 for declining pages vs. 472 for the rest -- decline is not confined to low-traffic pages, so there is real demand at stake.
3) 13,152 pages (43.8%) are both declining AND still pulling >=100 impressions/90d -- real, visible demand at risk, not noise on pages nobody sees.


## 4. Careful words: what I can and can't claim

**What this work can say:** observed and directional findings from FlyRank's own historical search
and analytics signals -- "these pages showed this pattern over this window," "this ranking method
recovers more of the declining pages in its top 50 than the hand-written rule does, on this
validation split." All of it is decision-support: a ranked list for a human reviewer to act on,
backed by reason codes they can inspect and override.

**What this work will never claim:**
- That a refresh *caused* a recovery -- that needs an actual experiment (a before/after test with a
  control), which this observational data alone cannot give.
- Anything about Google's ranking algorithm itself, or AI search citations/rankings -- the data only
  has FlyRank's own observed search and analytics numbers, never the algorithm's internals.
- That the ranked queue guarantees any specific page will improve -- it says "review this one
  first," not "this one will recover."
- Anything confident from the AI-referral columns, which are extremely sparse (30,177 rows with any
  AI session against ~78.8M daily rows in the full warehouse) -- those numbers can only support
  broad, cautious, directional observations, never a confident model.


In [4]:
has_ai = (df["ai_sessions_90d"] > 0).mean()
print(
    f"Check: share of pages in this starter slice with any AI-referred session at all: "
    f"{has_ai:.1%} -- a reminder to keep any AI-referral claim directional, not a confident model, "
    f"per the lane guide's warning about sparse AI-session data."
)


Check: share of pages in this starter slice with any AI-referred session at all: 6.4% -- a reminder to keep any AI-referral claim directional, not a confident model, per the lane guide's warning about sparse AI-session data.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.
